In [ ]:
# Import required libraries
import os, sys
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

# The reconstruction modules (reconstruction_grid / spatiotemporal_sampling) live in the
# deep-time-mining repo and internally load Reconstruction.py via Path.cwd()/backend/app-logic,
# so we must RUN FROM that repo root.
DTM_ROOT = Path("<PATH_TO>/deep-time-mining")
os.chdir(DTM_ROOT)
backend_path = DTM_ROOT / "backend" / "app-logic"
if str(backend_path) not in sys.path:
    sys.path.insert(0, str(backend_path))

# PROJ database for geopandas/pyproj (active env)
import pyproj
_proj = Path(sys.prefix) / "share" / "proj"
if _proj.exists():
    os.environ["PROJ_LIB"] = str(_proj); pyproj.datadir.set_data_dir(str(_proj))

# Import the reconstruction and sampling modules
import reconstruction_grid as rg

In [ ]:

## INPUT


#### This is from PlateModel


# Define paths to plate model files
data_folder = Path.cwd() / "backend" / "app-logic" / "Data"

rotation_file = "<DATA_ROOT>/Raw/plate_model/combined/CombinedRotations.rot"
partitioning_file="<DATA_ROOT>/Raw/plate_model/StaticGeometries/StaticPolygons/Global_EarthByte_GPlates_PresentDay_StaticPlatePolygons.shp"

# Configure reconstruction parameters
start_time = 0      # Present day
end_time = 170      # 250 Ma
timestep = 1       # Every 10 Ma




### This is the deposit data. We will reconstruct the locations of these deposits back in time, and then sample them onto a grid to create a raster dataset that can be used for modeling.
deposits_file="<DATA_ROOT>/CopperLithium/USA/ShapeFiles/pcu_deps_pros.csv"
lon_col='longitude'
lat_col='latitude'
age_col='age_ma'


# Named copper deposits (PCU): 'name' (Safford, Glacier Peak, ...), longitude/latitude, age_ma.
# Figure 4 searches deposits by name, so reconstruct THIS set.
deposit_data = pd.read_csv(deposits_file)
deposit_data = deposit_data.dropna(subset=[lon_col, lat_col])

# (old test source had only 8 unnamed points A-H, can't drive Figure 4):
# probably_deposits_file = "/Volumes/.../Raw/Vectors/Deposits/ProbableDeposits.shp"
# deposit_data = gpd.read_file(probably_deposits_file)
# deposit_data[lon_col] = deposit_data.geometry.x; deposit_data[lat_col] = deposit_data.geometry.y
# deposit_data = pd.DataFrame(deposit_data.drop(columns="geometry"))


# Reconstruct the point data
xr_deposits = rg.reconstruct_point_data(
    point_data=deposit_data,
    rotation_file=rotation_file,
    partitioning_file=partitioning_file,
    start_time=start_time,
    end_time=end_time,
    timestep=timestep,
    lon_col=lon_col,
    lat_col=lat_col,
    preserve_attributes=True,
    # from_age_col=age_col
)

# Display the result
print("\n" + "="*60)
print("Reconstruction complete!")
print(xr_deposits)

In [ ]:
# Sample spatiotemporal CSV predictions onto reconstruction grid
import spatiotemporal_sampling as sts

csv_path = "<DATA_ROOT>/Paper/Zenodo_DataBundle/data/outputs/spatiotemporal_grid_predictions_latest.csv"

# Quick preview of CSV structure
# df = pd.read_csv(csv_path)
# df.head()

In [ ]:
# Sample ALL predictions from CSV onto your deposits
# This transfers values via nearest-neighbor at each time step

xr_enriched = sts.sample_scattered_to_grid(
    xr_dataset=xr_deposits,
    csv_path=csv_path,
    lon_col='lon',
    lat_col='lat',
    time_col='age (Ma)',
    value_cols=None,  # None = transfer all numeric columns
    max_distance_deg=5.0,  # Only sample within 10 degrees (~1100 km)
    add_distance=True,
    verbose=True
)

print("\n" + "="*60)
print("✓ Enriched dataset with CSV predictions:")
print(xr_enriched)

In [ ]:
explorer = sts.ReconstructionExplorer(xr_enriched, verbose=True)

In [ ]:
lon,lat=-111.068,	31.984

In [ ]:
result = explorer.search(lon=lon, lat=lat)

In [ ]:
explorer.list_variables()

In [ ]:
# explorer.plot_variables(
#     variables=[
#         "Prospectivity Score",
#         "crustal_thickness_mean (m)",
#         "carbonate_thickness (m)",
#         'total_precipitation (km)'
#     ],
#     query="D"
# )

In [ ]:
# ============================================================
# FIGURE 4 — deposit property evolution (publication, large fonts)
# edit `names`, `properties`, `window`, and the FS{} font sizes below
# ============================================================

import re

def format_label(label):
    if not isinstance(label, str):
        return label
    label = label.replace("_", " ").strip()
    parts = label.split()
    if len(parts) > 1:
        if "(" in parts[-1]:
            main = " ".join(parts[:-2]); last = " ".join(parts[-2:])
        else:
            main = " ".join(parts[:-1]); last = parts[-1]
        label = main + "\n" + last
    return label[0].upper() + label[1:]


# ---- font sizes (EDIT to taste) --------------------------------------------
FS = dict(ylabel=16, xlabel=17, tick=14, legend=13.5, title=20, panel=17)

def plot_all_deposits_publication(explorer, deposit_names, properties, window=7,
                                  figsize=(15, 13), cmap="Set2", xmax=100,
                                  legend_on_all=False):
    """Publication multi-panel: properties (rows) x deposits (coloured lines), large fonts."""
    import matplotlib.pyplot as plt
    from matplotlib import cm
    import numpy as np, pandas as pd

    def extract_value(val):
        if isinstance(val, pd.Series):
            return val.iloc[0] if len(val) > 0 else None
        return val

    colors = cm.get_cmap(cmap, max(len(deposit_names), 3))
    deposit_colors = [colors(i) for i in range(len(deposit_names))]
    n_props = len(properties)
    fig, axes = plt.subplots(n_props, 1, figsize=figsize, sharex=True)
    if n_props == 1:
        axes = [axes]
    age_info = {}

    for idx, name in enumerate(deposit_names):
        try:
            df1 = explorer.search(query=name)
            df2 = explorer.get_dataframe(query=name)
            age_info.setdefault(name, {
                "age_ma": extract_value(df1.get("age_ma", None)),
                "age_range": extract_value(df1.get("age_range", None)),
                "color": deposit_colors[idx]})
            tcol = "time (Ma)" if "time (Ma)" in df2.columns else "time"
            for pi, prop in enumerate(properties):
                if prop not in df2.columns:
                    continue
                d = df2[[tcol, prop]].dropna().sort_values(tcol).copy()
                if len(d) == 0:
                    continue
                d["sm"] = d[prop].rolling(window=window, center=True, min_periods=1).mean()
                axes[pi].plot(d[tcol], d["sm"], linewidth=2.8, color=deposit_colors[idx],
                              label=name, alpha=0.9)
        except Exception as e:
            print(f"Warning: could not plot {name}: {e}")
            continue

    for pi, prop in enumerate(properties):
        ax = axes[pi]
        ax.set_ylabel(format_label(prop), fontsize=FS["ylabel"], fontweight="bold")
        ax.grid(alpha=0.3, linestyle="--", linewidth=0.6)
        ax.tick_params(labelsize=FS["tick"])
        ax.text(-0.015, 1.04, f"({chr(97+pi)})", transform=ax.transAxes,
                fontsize=FS["panel"], fontweight="bold", va="bottom", ha="right")
        if legend_on_all or pi == 0:
            ax.legend(loc="best", framealpha=0.9, fontsize=FS["legend"],
                      ncol=min(len(deposit_names), 3))
        if "Prospectivity Score" in prop:
            for name, info in age_info.items():
                age_ma, age_range, color = info["age_ma"], info["age_range"], info["color"]
                if age_ma is not None and not pd.isna(age_ma):
                    ax.axvline(float(age_ma), color=color, linestyle=":", linewidth=2.6, alpha=0.7)
                    if isinstance(age_range, (list, tuple)) and len(age_range) == 2:
                        ax.axvspan(float(age_range[0]), float(age_range[1]), color=color, alpha=0.15, linewidth=0)
                    elif isinstance(age_range, str):
                        parts = re.split(r"[-\u2013\u2014]", age_range.replace("Ma", "").strip())
                        if len(parts) == 2:
                            ax.axvspan(float(parts[0].strip()), float(parts[1].strip()),
                                       color=color, alpha=0.15, linewidth=0)

    axes[-1].set_xlabel("Time (Ma)", fontsize=FS["xlabel"], fontweight="bold")
    axes[-1].set_xlim(xmax, 0)   # older ages on the left -> present on the right
    fig.suptitle("Spatiotemporal evolution of copper deposits",
                 fontsize=FS["title"], fontweight="bold", y=0.997)
    plt.tight_layout()
    return fig, axes, age_info

# ---- Figure 4: select deposits + properties, then plot ----------------------
names = ["Safford", "Glacier Peak"]            # add/remove deposits here
properties = [
    "Prospectivity Score",
    "crustal_thickness_mean (m)",
    "subducted_carbonates_volume (m)",
    "convergence_rate_parallel (cm/yr)",
]
fig, axes, age_info = plot_all_deposits_publication(
    explorer=explorer, deposit_names=names, properties=properties,
    window=7, figsize=(15, 13), cmap="Set2", xmax=100)

import os
_out = "<DATA_ROOT>/Figure/keyfigures"
os.makedirs(_out, exist_ok=True)
fig.savefig(f"{_out}/Figure4_deposit_evolution.png", dpi=300, bbox_inches="tight")
fig.savefig(f"{_out}/Figure4_deposit_evolution.svg", bbox_inches="tight")

print("\nDeposit age information")
print("=" * 60)
for name, info in age_info.items():
    print(f"{name:20s} age: {info['age_ma']} Ma, range: {info['age_range']}")

In [ ]:
explorer.plot_variables(
    variables=[
        "Prospectivity Score",
        "crustal_thickness_mean (m)",
        "carbonate_thickness (m)",
        'total_precipitation (km)'
    ],
    query=result["name"]
)

In [ ]:
def plot_smoothed_prospectivity(
    df,
    score_col="Prospectivity Score",
    window=7,
    figsize=(8, 3),
):
    tcol = "time (Ma)" if "time (Ma)" in df.columns else "time"
    if tcol not in df.columns:
        raise ValueError("No time column found. Expected 'time (Ma)' or 'time'.")
    if score_col not in df.columns:
        raise ValueError(f"Column '{score_col}' not found in dataframe.")

    d = df[[tcol, score_col]].dropna().sort_values(tcol).copy()
    d[f"{score_col} (smoothed)"] = (
        d[score_col].rolling(window=window, center=True, min_periods=1).mean()
    )

    plt.figure(figsize=figsize)
    # plt.plot(d[tcol], d[score_col], alpha=0.35, label="Raw")
    plt.plot(d[tcol], d[f"{score_col} (smoothed)"], linewidth=2)
    plt.xlabel(tcol)
    plt.ylabel(score_col)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()

    return d

names=["Safford","Morenci-Metcalf","Continental/Butte","Bingham", "Chino"]
i=4
df1=explorer.search(query=names[i])
df2=explorer.get_dataframe(query=names[i])
print(df1['age_ma'], df1['age_range'])
plot_smoothed_prospectivity(df2, score_col="Prospectivity Score", window=5)
plot_smoothed_prospectivity(df2, score_col="crustal_thickness_mean (m)", window=5)
plot_smoothed_prospectivity(df2, score_col="carbonate_thickness (m)", window=5)

In [ ]:
names=["Safford","Morenci-Metcalf","Continental/Butte","Bingham", "Chino", "Ray"]
i=5
df1=explorer.search(query=names[i])
df1['latitude'],df1['longitude']

In [ ]:
names=["Safford","Morenci-Metcalf","Continental/Butte","Bingham", "Chino"]
i=0
df1=explorer.search(query=names[i])
df1['latitude'],df1['longitude']

In [ ]:
names=["Safford","Morenci-Metcalf","Continental/Butte","Bingham", "Chino"]
i=4
df1=explorer.search(query=names[i])
df2=explorer.get_dataframe(query=names[i])
print(df1['age_ma'], df1['age_range'])
plot_smoothed_prospectivity(df2, score_col="Prospectivity Score", window=5)
plot_smoothed_prospectivity(df2, score_col="crustal_thickness_mean (m)", window=5)
plot_smoothed_prospectivity(df2, score_col="subducted_carbonates_volume (m)", window=5)


In [ ]:
# (superseded) Figure 4 is generated by the large-font cell above.


In [ ]:
explorer.plot_variables(
    variables=[
        "Prospectivity Score",
        "crustal_thickness_mean (m)",
        "carbonate_thickness (m)",
        'total_precipitation (km)'
    ],
    query="Chino"
)

In [ ]:
explorer.plot_variables(
    variables=[
        "Prospectivity Score",
        "crustal_thickness_mean (m)",
        "carbonate_thickness (m)",
        'total_precipitation (km)'
    ],
    query=result["name"]
)

In [ ]:
# Plot multiple variables for one deposit
explorer.plot_variables(
    variables=[
        "Prospectivity Score",
        "crustal_thickness_mean (m)",
        "slab_flux (kg/s)"
    ],
    query="Safford"  # Gets first porphyry deposit
)